# 03 · The unified token-budget scheduler

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/nilesh-salpe/visual-ai/blob/main/build-it/vllm/03_token_budget_scheduler.ipynb) &nbsp; 📖 Concept deck: [The scheduler: continuous batching and chunked prefill](https://nilesh-salpe.github.io/visual-ai/vllm/04-scheduler/) &nbsp; ⏱️ Suggested time: **40 min**

**You will build:** a step-by-step simulation of vLLM V1's scheduler: a per-step token budget shared by decodes and (chunked) prefills, running requests served first, a long-prefill threshold, and preemption by recomputation when KV blocks run out.

**How to use this notebook**
1. Run the **Setup** cell.
2. For each task, fill in the `TODO` in the code cell, then run the **Tests** cell under it. You'll see ⏳ (not done yet), ✅ (passed) or ❌ (failed, with the reason).
3. Stuck? The **Reference solutions** at the bottom are collapsed. Try first.

Everything runs on CPU with NumPy, so no GPU is needed.

In [ ]:
# @title Setup: run this first
import numpy as np, math, time, itertools, collections
np.set_printoptions(precision=4, suppress=True)

def check(name, test, impl):
    """Run `test(impl)` and report: ⏳ not implemented, ✅ passed, ❌ failed with the reason."""
    try:
        test(impl)
        print(f"✅ {name}: all tests passed")
    except NotImplementedError:
        print(f"⏳ {name}: not implemented yet. Fill in the TODO in the cell above, then re-run this cell.")
    except AssertionError as e:
        print(f"❌ {name}: {e}")


import math
from dataclasses import dataclass, field
BLOCK = 16
@dataclass
class Req:
    rid: int
    prompt: int            # prompt tokens
    out: int               # tokens to generate
    computed: int = 0      # num_computed_tokens
    generated: int = 0
    blocks: int = 0
    preemptions: int = 0
    def total(self):       # tokens that should have KV once caught up
        return self.prompt + self.generated

## Task 1: One scheduling step

Implement `step(running, waiting, budget, free_blocks, long_prefill=0)` following vLLM V1:
1. For each **running** request in order: it wants `total() - computed` new tokens (prefill chunks or 1 decode token); cap by `long_prefill` (if > 0) and by the remaining budget. If it needs more blocks than are free, **preempt** the *last* running request (free its blocks, set `computed = 0`, move it to the front of `waiting`) and retry; if it is itself the last, preempt it.
2. Then admit **waiting** requests in order while budget and blocks remain.
Return `(scheduled, free_blocks)` with `scheduled = {rid: num_tokens}`. Blocks needed = ⌈(computed + n) / 16⌉ − blocks.

In [ ]:
def step(running, waiting, budget, free_blocks, long_prefill=0):
    scheduled = {}
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 1: run after implementing
def _test_step(f):
    run = [Req(0, 10, 5, computed=10, generated=1, blocks=1)]; wait = [Req(1, 100, 5)]
    s, free = f(run, wait, budget=64, free_blocks=100)
    assert s == {0: 1, 1: 63}, f"decode first (1 token), then a 63-token prefill chunk; got {s}"
    assert run[-1].rid == 1 and not wait, "an admitted request moves to running"
    run = [Req(0, 16, 5, computed=16, generated=1, blocks=1), Req(1, 40, 5, computed=40, generated=1, blocks=3)]
    s, free = f(run, [], budget=64, free_blocks=0)
    assert 0 in s and 1 not in s, "request 0 needs a new block, so the last running request is preempted"

check("step", _test_step, step)

## Task 2: Run the engine loop

Repeatedly call your `step`, then **apply** it: add scheduled tokens to `computed`; a request whose `computed` reaches `total()` samples one token (`generated += 1`); requests with `generated == out` finish and free their blocks. Return `(steps, total_preemptions, tokens_per_step)`.

In [ ]:
def simulate(reqs, budget=256, num_blocks=200, long_prefill=0, max_steps=10_000):
    running, waiting, free = [], list(reqs), num_blocks
    # TODO
    raise NotImplementedError

In [ ]:
# @title Tests for Task 2: run after implementing
def _test_simulate(f):
    reqs = [Req(i, 200 + 37 * i, 20 + 5 * i) for i in range(8)]
    steps, pre, tps = f([Req(r.rid, r.prompt, r.out) for r in reqs], budget=256, num_blocks=400)
    assert pre == 0, "plenty of blocks: no preemptions"
    assert steps < 200
    steps2, pre2, _ = f([Req(r.rid, r.prompt, r.out) for r in reqs], budget=256, num_blocks=40)
    assert pre2 > 0, "too few blocks: requests get preempted and recomputed"

check("simulate", _test_simulate, simulate)

---
## Reference solutions
Try the tasks first. Each cell below is collapsed in Colab: double-click it (or click "Show code") to reveal the solution. Running it also checks the reference against the same tests.

In [ ]:
# @title Solution for Task 1: One scheduling step
def step_solution(running, waiting, budget, free_blocks, long_prefill=0):
    scheduled = {}
    i = 0
    while i < len(running) and budget > 0:
        r = running[i]
        n = r.total() - r.computed
        if long_prefill > 0: n = min(n, long_prefill)
        n = min(n, budget)
        need = math.ceil((r.computed + n) / BLOCK) - r.blocks
        while need > free_blocks:
            victim = running.pop()
            free_blocks += victim.blocks; victim.blocks = 0; victim.computed = 0; victim.preemptions += 1
            scheduled.pop(victim.rid, None); waiting.insert(0, victim)
            if victim is r: break
        else:
            r.blocks += need; free_blocks -= need; scheduled[r.rid] = n; budget -= n; i += 1
            continue
        break
    while waiting and budget > 0:
        r = waiting[0]
        n = r.total() - r.computed
        if long_prefill > 0: n = min(n, long_prefill)
        n = min(n, budget)
        need = math.ceil((r.computed + n) / BLOCK) - r.blocks
        if need > free_blocks: break
        waiting.pop(0); running.append(r)
        r.blocks += need; free_blocks -= need; scheduled[r.rid] = n; budget -= n
    return scheduled, free_blocks

check("step (reference)", _test_step, step_solution)

In [ ]:
# @title Solution for Task 2: Run the engine loop
def simulate_solution(reqs, budget=256, num_blocks=200, long_prefill=0, max_steps=10_000, step_fn=None):
    step_fn = step_fn or step_solution
    running, waiting, free = [], list(reqs), num_blocks
    steps, tps = 0, []
    while (running or waiting) and steps < max_steps:
        s, free = step_fn(running, waiting, budget, free, long_prefill)
        steps += 1; tps.append(sum(s.values()))
        for r in list(running):
            if r.rid in s:
                r.computed += s[r.rid]
                if r.computed == r.total():
                    r.generated += 1
                    if r.generated == r.out:
                        running.remove(r); free += r.blocks; r.blocks = 0
    return steps, sum(r.preemptions for r in reqs), tps

check("simulate (reference)", _test_simulate, simulate_solution)

---
## Experiments
**Token budget vs throughput and smoothness.** A larger `max_num_batched_tokens` finishes in fewer steps but makes each step longer (higher inter-token latency for running requests). A long-prefill threshold caps how much of a step one huge prompt can take, leaving room for other prefills.

In [ ]:
def make(): return [Req(i, 2000 if i == 0 else 100 + 20 * i, 30) for i in range(12)]
for budget in [128, 512, 2048]:
    st, pre, tps = simulate_solution(make(), budget=budget, num_blocks=1000)
    print(f"budget {budget:5d}: {st:4d} steps, max tokens in one step {max(tps):5d}, preemptions {pre}")
for lp in [0, 256]:
    biggest = []
    def rec(*a, **k):
        s, free = step_solution(*a, **k); biggest.append(max(s.values(), default=0)); return s, free
    st, pre, tps = simulate_solution(make(), budget=2048, num_blocks=1000, long_prefill=lp, step_fn=rec)
    print(f"budget 2048, long_prefill_token_threshold {lp:3d}: {st} steps, largest chunk given to one request in a step: {max(biggest)} tokens")

---
## Takeaways
* vLLM V1's scheduler has no prefill or decode phase: each step hands out a token budget so every request's computed tokens catch up with its known tokens.
* Running requests are served first; new requests fill the remaining budget, with long prompts split into chunks (chunked prefill is on by default).
* When KV blocks run out, the most recently admitted running request is preempted: its blocks are freed and it is recomputed later.